# Resultados de los experimentos

Tablas con **accuracy**, **F1** y **métricas por clase** (P2P, C2P, P2C) de todos los modelos entrenados en los dos enfoques:

* **End-to-end** (`results/end_to_end/<caso>/<modelo>_<decoder>_seed<S>_<config_id>/metrics.json`)
* **Por partes** (`results/EnfoquePorPartes/clasificador_ann/<embedding>_ann_seed<S>_<config_id>/metrics.json`): solo el clasificador ANN guarda métricas de clasificación; los GNN que generan los embeddings no.

Este notebook solo **lee** los `metrics.json`; no entrena ni modifica nada. Cada modelo y decoder entrenado aparece en las tablas, no solo el mejor. `config_id` identifica la combinación de hiperparámetros y clase del modelo (capas): si cambia, es otra corrida.

In [1]:
import re, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# raíz del repositorio (para importar modules.experiment)
REPO = Path.cwd()
if not (REPO / "modules").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from modules.experiment import config_id

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

## Configuración

In [2]:
EXPERIMENTO = Path("/media/vale/KINGSTON/as-level-internet-graph/experimentos/experimento7")
GRAFOS = ["2026"]        # solo grafos cuyo nombre/ruta contenga alguna de estas cadenas ([] = todos)
RAPIDO = False            # True: lee GRAFO_DIR/prueba_rapida/results (prueba de humo); False: GRAFO_DIR/results
SEED = None              # None = todas las semillas
CLASES = ["P2P", "C2P", "P2C"]
EXPORTAR = False         # True: guarda las tablas como CSV en EXPERIMENTO/resultados_gnn/tablas/

## Carga de los `metrics.json`

In [3]:
HIPERPARAMETROS = ["lr", "hidden_dim", "out_dim", "epochs", "dropout", "weight_decay", "num_heads"]


def _fila(m, enfoque, grafo, carpeta):
    extra, cfg = m.get("extra") or {}, m.get("config") or {}
    clase = extra.get("clase")
    capas = re.search(r"(\d)L", clase or m["modelo"])
    tiempo = extra.get("tiempo_s")
    fila = {
        "grafo": grafo, "enfoque": enfoque,
        "caso": extra.get("caso_embedding") if enfoque == "por_partes" else m["caso"],
        "modelo": m["modelo"], "decoder": m["decoder"], "seed": m["seed"],
        "clase": clase or extra.get("clase_embedding"),
        "n_capas": int(capas.group(1)) if capas else np.nan,
        "config_id": m.get("config_id") or config_id(cfg, extra),
        **{h: cfg.get(h) for h in HIPERPARAMETROS},
        "accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "weighted_f1": m["weighted_f1"],
        "n_test": m["n_test"],
        "tiempo_s": tiempo, "epocas": extra.get("epocas"),
        "tiempo_embedding_s": extra.get("tiempo_embedding_s"),
        "tiempo_total_s": extra.get("tiempo_total_s", tiempo),
        "carpeta": str(carpeta),
    }
    for c in CLASES:
        r = (m.get("reporte") or {}).get(c, {})
        fila[f"{c}_precision"], fila[f"{c}_recall"], fila[f"{c}_f1"] = (
            r.get("precision"), r.get("recall"), r.get("f1-score"))
    fila["matriz_confusion"] = m.get("matriz_confusion")
    return fila


def cargar_metricas():
    filas, sin_resultados = [], []
    grafos = sorted(p for p in (EXPERIMENTO / "grafos").glob("*/*/*") if p.is_dir())
    for g in grafos:
        if GRAFOS and not any(f in str(g.relative_to(EXPERIMENTO / "grafos")) for f in GRAFOS):
            continue
        base = g / ("prueba_rapida" if RAPIDO else "") / "results"
        patrones = (("end_to_end", "end_to_end/*/*/metrics.json"),
                    ("por_partes", "EnfoquePorPartes/clasificador_ann/*/metrics.json"))
        n = 0
        for enfoque, patron in patrones:
            for f in sorted(base.glob(patron)):
                m = json.loads(f.read_text(encoding="utf-8"))
                if SEED is None or m["seed"] == SEED:
                    filas.append(_fila(m, enfoque, g.name, f.parent)); n += 1
        if n == 0:
            sin_resultados.append(g.name)
    if sin_resultados:
        print(f"Sin resultados ({'prueba_rapida' if RAPIDO else 'results'}) en: {', '.join(sin_resultados)}")
    return pd.DataFrame(filas)


df = cargar_metricas()
print(f"{len(df)} corridas cargadas")
if df.empty:
    raise SystemExit("No hay resultados: revisa EXPERIMENTO, GRAFOS y RAPIDO.")
if df.groupby(["grafo", "enfoque", "caso", "modelo", "decoder", "seed"]).config_id.nunique().gt(1).any():
    print("Aviso: hay corridas del mismo modelo con distinta configuración (config_id); aparecen como filas separadas.")
df.groupby(["enfoque", "caso"]).size().unstack(0).fillna(0).astype(int)

Sin resultados (results) en: 20260916T0800
0 corridas cargadas


SystemExit: No hay resultados: revisa EXPERIMENTO, GRAFOS y RAPIDO.

/home/vale/Escritorio/GIT/LABORATORIO/gnn-as-relationships/env310/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Utilidades de formato

Las métricas se muestran en porcentaje; el valor máximo de cada columna de métricas aparece resaltado.

In [ ]:
PCT = ["accuracy", "macro_f1", "weighted_f1"]
TIEMPOS = ["tiempo_s", "tiempo_embedding_s", "tiempo_total_s"]
exportadas = {}


def mostrar(tabla, nombre, metricas=None, tiempos=()):
    """Muestra una tabla con formato (porcentajes, segundos, máximo resaltado) y opcionalmente la exporta."""
    exportadas[nombre] = tabla
    metricas = [c for c in (metricas if metricas is not None else PCT) if c in tabla.columns]
    fmt = {c: "{:.2%}" for c in metricas}
    fmt.update({c: "{:.1f}" for c in tiempos if c in tabla.columns})
    if "epocas" in tabla.columns:
        fmt["epocas"] = "{:.0f}"
    estilo = tabla.style.format(fmt, na_rep="–")
    if metricas:
        estilo = estilo.highlight_max(subset=metricas, color="#cfe8cf")
    display(estilo)


def titulo(texto):
    display(Markdown(texto))

## 1. Resumen general (todos los modelos, ordenados por macro-F1)

In [ ]:
COLS_E2E = ["grafo", "caso", "modelo", "decoder", "n_capas", "accuracy", "macro_f1", "weighted_f1",
            "tiempo_s", "epocas", "config_id"]
COLS_PP = ["grafo", "caso", "modelo", "clase", "accuracy", "macro_f1", "weighted_f1",
           "tiempo_s", "tiempo_embedding_s", "tiempo_total_s", "config_id"]

for enfoque, cols in (("end_to_end", COLS_E2E), ("por_partes", COLS_PP)):
    t = df[df.enfoque == enfoque]
    titulo(f"### {enfoque.replace('_', '-')} ({len(t)} corridas)")
    if t.empty:
        print("Sin resultados."); continue
    mostrar(t[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True),
            f"resumen_{enfoque}", tiempos=TIEMPOS)

## 2. End-to-end: un caso a la vez

* `caso0`, `caso1`: link prediction/clasificación full-batch (decoders MLP y Bilinear)
* `caso2_neighbor`: muestreo de vecinos · `caso3_cluster`: ClusterGCN · `caso4_random`: features aleatorias
* `comparacion`: GCN/GraphSAGE/GAT de 2 y 3 capas con ambos decoders

In [ ]:
ORDEN_E2E = ["caso0", "caso1", "caso2_neighbor", "caso3_cluster", "caso4_random", "comparacion"]
e2e = df[df.enfoque == "end_to_end"]
for caso in ORDEN_E2E + sorted(set(e2e.caso) - set(ORDEN_E2E)):
    t = e2e[e2e.caso == caso]
    titulo(f"### {caso}")
    if t.empty:
        print("Sin resultados."); continue
    mostrar(t[COLS_E2E[:1] + COLS_E2E[2:]].sort_values(["modelo", "decoder"]).reset_index(drop=True),
            f"e2e_{caso}", tiempos=TIEMPOS)

## 3. Por partes: un caso de origen a la vez

Cada fila es un embedding clasificado con la ANN. `tiempo_s` es el entrenamiento de la ANN, `tiempo_embedding_s` el del modelo que generó el embedding y `tiempo_total_s` la suma (vacío si el embedding es anterior al registro de tiempos).

* Caso0/Caso1: link prediction (decoders DotProduct/MLP) · Caso3: regresión de grado · Caso4: regresión de PageRank · Caso5: DeepWalk y BGP2Vec

In [ ]:
pp = df[df.enfoque == "por_partes"]
for caso in sorted(pp.caso.dropna().unique()):
    t = pp[pp.caso == caso]
    titulo(f"### {caso}")
    mostrar(t[COLS_PP[:1] + COLS_PP[2:]].sort_values("macro_f1", ascending=False).reset_index(drop=True),
            f"partes_{caso}", tiempos=TIEMPOS)
if pp.empty:
    print("Sin resultados.")

## 4. Métricas por clase (P2P, C2P, P2C)

Precision, recall y F1 de **cada** modelo y decoder entrenado, por caso.

In [ ]:
def por_clase(t, indice):
    t = t.set_index(indice)
    cols = pd.MultiIndex.from_tuples([(c, m) for c in CLASES for m in ("precision", "recall", "f1")],
                                     names=["clase", "métrica"])
    out = pd.DataFrame({(c, m): t[f"{c}_{m}"] for c, m in cols})
    out.columns = cols
    return out


def mostrar_por_clase(t, nombre, indice):
    tabla = por_clase(t, indice)
    exportadas[nombre] = tabla
    display(tabla.style.format("{:.2%}", na_rep="–").highlight_max(axis=0, color="#cfe8cf"))


for enfoque, indice in (("end_to_end", ["caso", "modelo", "decoder", "config_id"]),
                        ("por_partes", ["caso", "modelo", "config_id"])):
    titulo(f"### {enfoque.replace('_', '-')}")
    t = df[df.enfoque == enfoque]
    if t.empty:
        print("Sin resultados."); continue
    for caso, tc in t.groupby("caso"):
        titulo(f"**{caso}**")
        mostrar_por_clase(tc,
                          f"por_clase_{enfoque}_{caso}", indice[1:])

## 5. End-to-end vs por partes: mejor modelo y tiempo

In [ ]:
def mejores(t):
    return t.sort_values("macro_f1", ascending=False).groupby(["grafo", "enfoque", "caso"], as_index=False).head(1)


mejor = mejores(df)[["grafo", "enfoque", "caso", "modelo", "decoder", "n_capas", "accuracy", "macro_f1",
                     "tiempo_s", "tiempo_embedding_s", "tiempo_total_s", "config_id"]]
mostrar(mejor.sort_values(["grafo", "enfoque", "macro_f1"], ascending=[True, True, False]).reset_index(drop=True),
        "mejores_por_caso", tiempos=TIEMPOS)

## 6. Matriz de confusión del mejor modelo de cada enfoque

In [ ]:
for (grafo, enfoque), t in df.groupby(["grafo", "enfoque"]):
    m = t.sort_values("macro_f1", ascending=False).iloc[0]
    if m.matriz_confusion is None:
        continue
    cm = np.array(m.matriz_confusion, dtype=float)
    cmp = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    im = ax.imshow(cmp, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3), CLASES); ax.set_yticks(range(3), CLASES)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.set_title(f"{enfoque} · {m.caso} · {m.modelo} {m.decoder}\n{grafo}", fontsize=9)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cmp[i, j]:.0%}\n({int(cm[i, j]):,})", ha="center", va="center",
                    fontsize=8, color="white" if cmp[i, j] > 0.5 else "black")
    fig.tight_layout(); plt.show()

## 7. Evolución entre grafos (solo si hay más de uno)

macro-F1 de cada modelo en cada grafo.

In [ ]:
if df.grafo.nunique() > 1:
    p = df.pivot_table(index=["enfoque", "caso", "modelo", "decoder"], columns="grafo", values="macro_f1",
                       aggfunc="max")
    exportadas["macro_f1_por_grafo"] = p
    display(p.style.format("{:.2%}", na_rep="–"))
else:
    print("Hay un solo grafo cargado.")

## 8. Exportar tablas (opcional)

In [ ]:
if EXPORTAR:
    destino = EXPERIMENTO / "resultados_gnn" / "tablas"
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, tabla in exportadas.items():
        tabla.to_csv(destino / f"{nombre}.csv")
    print(f"{len(exportadas)} tablas guardadas en {destino}")
else:
    print("EXPORTAR = False: no se guardó nada.")